# Essential Dynamics Analysis — Backbone

Eigenvalue spectrum, 2D projection free-energy histogram, and representative low-energy structures for the aqueous cationic peptide GAG⁺.

Essential Dynamics (ED — equivalent to PCA of an MD trajectory) reduces the production trajectory to a small number of collective backbone coordinates (eigenvectors) that capture most of the conformational fluctuation, following M. Monti, M. Stener, M. Aschi, *J. Comput. Chem.* **43** (2022) 2023–2036 (https://doi.org/10.1002/jcc.27001).

## 1. Eigenvalues of the covariance matrix

`gmx covar` diagonalizes the covariance matrix of backbone atomic-position fluctuations (after a least-squares fit removes overall translation/rotation), giving eigenvectors and their eigenvalues. Eigenvectors are ordered from largest to smallest eigenvalue, so the first few describe the largest-amplitude collective motions sampled during the trajectory — these are the essential coordinates.

For the 2D essential-plane projection used later in this notebook to be a meaningful representation of the dominant conformational motions, the first two eigenvectors should capture a substantial share of the total fluctuation: (λ₁+λ₂)/Σᵢλᵢ, printed by the cell below. There's no universal threshold this has to clear — just enough to trust that the 2D landscape reflects real dominant motions rather than an arbitrary slice.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Load data, skipping comment lines starting with # or @
x, y = [], []
filename = "eigenval.xvg"

with open(filename, "r") as f:
    for line in f:
        if line.startswith(("#", "@")):
            continue
        cols = line.split()
        if len(cols) >= 2:
            x.append(float(cols[0]))
            y.append(float(cols[1]))

y = np.array(y)
pct_top2 = 100 * y[:2].sum() / y.sum()
print(f"First 2 eigenvalues cover {pct_top2:.2f}% of the total variance")

plt.figure(figsize=(8, 5))
plt.plot(x, y, color="blue", linestyle="-", marker="o", markersize=6, linewidth=1.5)
plt.title(r"Eigenvalues of the GAG$^+$ backbone covariance matrix", fontsize=16)
plt.xlabel("Atomic index", fontsize=16)
plt.ylabel("Eigenvalue (nm$^2$)", fontsize=16)
plt.xlim(left=0)
plt.tick_params(axis="both", which="major", length=6, width=1.2)
plt.yticks(fontsize=14)
plt.xticks(fontsize=14)

plt.tight_layout()
plt.savefig('eigenval.png', dpi=600)
plt.show()

## 2. 2D free-energy histogram + low-energy structures

Every trajectory frame's (PC-1, PC-2) is binned on a `bin_x` × `bin_y` grid, and each bin's population is converted into a relative free energy via the Boltzmann relation ΔGᵢ = -RT·ln(Pᵢ/Pmax), so the most-populated bin is assigned ΔG = 0 and everything else is relative to it. This is a *projected* relative free-energy landscape (population in the 2D essential plane), not an absolute free energy of the system — `temp` should match the production simulation's temperature. `bin_x = bin_y = 50` and `dime = 25001` because coordinates were saved every 2 ps over the 50 ns production run; bins with ΔG ≤ 2500 J/mol (2.5 kJ/mol) are treated as the low-energy, highly-populated regions and written to `low_en_structs.txt`.

In [ ]:
# --- Parameters formerly read from header-isto — set these manually for your run ---
dime = 25001    # number of (proj1, proj2) data points in 2dproj.xvg
temp = 303      # temperature of the MD simulation (K)
bin_x = 50      # number of bins along projection 1
bin_y = 50      # number of bins along projection 2

In [ ]:
import math

R = 8.314  # gas constant, J/(mol K)

# Read (p1, p2) pairs from 2dproj.xvg, skipping xvg-style comment lines (# or @)
p1, p2 = np.empty(dime), np.empty(dime)
i = 0
with open("2dproj.xvg") as f:
    for line in f:
        if line.startswith(("#", "@")):
            continue
        cols = line.split()
        if len(cols) < 2:
            continue
        p1[i], p2[i] = float(cols[0]), float(cols[1])
        i += 1
        if i == dime:
            break
assert i == dime, f"dime={dime} but only {i} data rows were found in 2dproj.xvg"

prob = np.zeros((bin_x, bin_y))
energy = np.zeros((bin_x, bin_y))

min_x, max_x = p1.min(), p1.max()
min_y, max_y = p2.min(), p2.max()
step1 = (max_x - min_x) / bin_x
step2 = (max_y - min_y) / bin_y

for i in range(1, bin_x):
    in_x = (p1 >= min_x + step1 * i) & (p1 < min_x + step1 * (i + 1))
    for j in range(1, bin_y):
        in_y = (p2 >= min_y + step2 * j) & (p2 < min_y + step2 * (j + 1))
        prob[i - 1, j - 1] = np.count_nonzero(in_x & in_y)

max_prob = prob.max()
his = []
with open("2d_histo.txt", "w") as histo_out:
    axis1 = min_x - step1
    for i in range(1, bin_x + 1):
        axis1 += step1
        axis2 = min_y - step2
        for j in range(1, bin_y + 1):
            axis2 += step2
            p = prob[i - 1, j - 1]
            if p != 0.0:
                e = -R * temp * math.log(p / max_prob)
                energy[i - 1, j - 1] = e
                histo_out.write(f"{axis1 + step1 / 2:12.5f} {axis2 + step2 / 2:12.5f} {e:12.5f}\n")
                if e <= 2500.0:
                    his.append((axis1 + step1 / 2, axis2 + step2 / 2, e))

print(f"wrote 2d_histo.txt ({bin_x * bin_y} bins scanned, {len(his)} below the 2500 cutoff)")

In [ ]:
with open("low_en_structs.txt", "w") as out:
    for h1, h2, h3 in his:
        # nearest real trajectory frame to this bin center 
        j = np.argmin((p1 - h1) ** 2 + (p2 - h2) ** 2)
        time = 2 * j  # matches istogramma.f90's `2*j_fortran - 2`
        out.write(f"{time:10d} {p1[j]:12.5f} {p2[j]:12.5f} {h3:12.5f}\n")

print(f"wrote low_en_structs.txt ({len(his)} structures)")

## 3. Plot histogram + low-energy configurations

In [ ]:
x, y, z = np.loadtxt('./2d_histo.txt', unpack=True)
t, c1, c2, en = np.loadtxt('./low_en_structs.txt', unpack=True)

plt.tricontourf(x, y, z / 1000, levels=10, cmap='rainbow')
plt.plot(c1, c2, "o", markersize=4, color='black')
plt.xlabel('PC-1 (nm)')
plt.ylabel('PC-2 (nm)')
plt.xlim(min(x), max(x))
plt.ylim(min(y), max(y))
plt.legend(frameon=False)
cbar = plt.colorbar()
cbar.set_label(r'$\Delta G$ (kJ/mol)')
plt.title(r"Free-energy landscape in the essential plane of GAG$^+$")

plt.savefig("./2dhi.png", dpi=600)
plt.show()

## 4. Representative structures per cluster

In [ ]:
from sklearn.cluster import DBSCAN

time, cp1, cp2, dG = np.loadtxt("low_en_structs.txt", unpack=True)

# Clustering radius (nm) and minimum cluster size — tune if the cluster count below
# doesn't match what you see in the histogram plot above.
eps = 0.03
min_samples = 3

labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(np.column_stack([cp1, cp2]))
n_clusters = len(set(labels) - {-1})
print(f"found {n_clusters} clusters ({(labels == -1).sum()} unclustered points)")

basin_colors = ["red", "darkgreen", "blue"]
assert n_clusters <= len(basin_colors), "add more colors to basin_colors if you retune eps to get more clusters"
low_en_color = [basin_colors[l] if l != -1 else "gray" for l in labels]

n_reps = 5  # representative structures per cluster, spanning very low -> high energy
selected = []
for lbl in sorted(set(labels)):
    if lbl == -1:
        continue
    idx = np.where(labels == lbl)[0]
    idx = idx[np.argsort(dG[idx])]  # low -> high energy within this cluster
    pick = np.unique(np.linspace(0, len(idx) - 1, min(n_reps, len(idx))).round().astype(int))
    selected.extend((lbl, time[i], cp1[i], cp2[i], dG[i]) for i in idx[pick])

In [ ]:
with open("selected_structures.txt", "w") as out:
    out.write(f"{'cluster':>7} {'time':>10} {'proj1':>10} {'proj2':>10} {'dG(J/mol)':>12}\n")
    for lbl, t, c1, c2, e in selected:
        out.write(f"{lbl:7d} {t:10.0f} {c1:10.5f} {c2:10.5f} {e:12.2f}\n")

print(f"{'cluster':>7} {'time (ps)':>10} {'proj1':>10} {'proj2':>10} {'DeltaG (J/mol)':>12}")
for lbl, t, c1, c2, e in selected:
    print(f"{lbl:7d} {t:10.0f} {c1:10.5f} {c2:10.5f} {e:12.2f}")

In [ ]:
sel = np.array([(c1, c2) for _, _, c1, c2, _ in selected])

plt.figure(figsize=(6, 5))
for lbl in sorted(set(labels)):
    m = labels == lbl
    color = basin_colors[lbl] if lbl != -1 else "gray"
    plt.scatter(cp1[m], cp2[m], color=color, s=20, label=f"basin_{lbl}" if lbl != -1 else "unclustered")
plt.scatter(sel[:, 0], sel[:, 1], facecolors="none", edgecolors="black", s=120, linewidths=1.5, label="selected")
plt.xlabel("PC-1 (nm)")
plt.ylabel("PC-2 (nm)")
plt.legend()
plt.tight_layout()
plt.savefig('selected_low_en_confs.png',dpi=600)
plt.show()

You can now execute the extract_structures.sh script to print the low energy conformations belonging to different basins and plot them on vmd.
How do these conformations look like? Can you see any visual difference between structures belonging to basin 0, 1, and 2?

# 5. Additional structural analysis

The Essential Dynamics landscape identifies the main conformational states sampled by GAG⁺. We now complement this picture with additional structural descriptors and ask if the peptide's **compactness** and its **interaction with the surrounding water** relate to the specific low-energy representative structures already identified in Section 4.

These analyses use **MDAnalysis** on the centered production trajectory (`prod_c.xtc`) with the production topology (`prod.tpr`) — the same frames used for the Essential Dynamics analysis above, so each structural descriptor can be related back to the peptide's position in the 2D free-energy landscape.

In [ ]:
import MDAnalysis as mda

u = mda.Universe("prod.tpr", "prod_c.xtc")
assert len(u.trajectory) == len(p1), "trajectory frame count must match 2dproj.xvg (Section 2) for frame<->basin alignment"
print(f"{len(u.trajectory)} frames, {len(u.atoms)} atoms")

protein = u.select_atoms("protein")
backbone = u.select_atoms("protein and backbone")
protein_heavy = u.select_atoms("protein and not name H*")
water_O = u.select_atoms("resname SOL and name OW")
print(f"protein: {len(protein)} atoms, backbone: {len(backbone)}, heavy: {len(protein_heavy)}, water O: {len(water_O)}")

plot_stride = 1 #for less thick plots we can plot every x steps (e.g. 50)

In [ ]:
# Reuse the low-energy structures already loaded in Section 4 (time, cp1, cp2, dG) 
# so this always matches `labels`/`low_en_color` row-for-row.
low_en_time, low_en_p1, low_en_p2, low_en_dG = time, cp1, cp2, dG
low_en_time_ns = low_en_time / 1000

dt = u.trajectory[1].time - u.trajectory[0].time
low_en_frame = np.round(low_en_time / dt).astype(int)

# Shared time axis (ps and ns) for every time-series plot in this section — computed
# once here (frames are evenly spaced by dt) rather than separately per cell.
time_ps = np.arange(len(u.trajectory)) * dt
time_ns = time_ps / 1000

print(f"dt = {dt:.1f} ps/frame, {len(low_en_frame)} representative structures mapped to trajectory frames")

## 5.1 Ramachandran plot

The backbone dihedral angles φ (C(i-1)-N-CA-C) and ψ (N-CA-C-N(i+1)) directly define the pPII/β-strand distinction discussed throughout this notebook — both are named regions of (φ, ψ) space.

With only 3 residues, φ needs a *previous* residue's C and ψ needs a *next* residue's N, so the two termini (GLY1, GLY3) each have only one of the two angles — only the middle residue, **ALA2**, has a complete (φ, ψ) pair. `MDAnalysis.analysis.dihedrals.Ramachandran` figures this out on its own: passed the full `protein` selection, it automatically drops residues lacking a complete pair (with a warning) and returns angles for ALA2 only.

As with the other descriptors, the low-energy representative structures are marked (colored by cluster), plus rough reference points for the canonical pPII (φ≈-75°, ψ≈145°), beta-strand/extended (φ≈-130°, ψ≈130°), and RH alpha-helix (φ≈-60°, ψ≈-45°) regions — approximate textbook locations, not a rigorous boundary, just a visual anchor.

In [ ]:
from MDAnalysis.analysis.dihedrals import Ramachandran

rama = Ramachandran(protein).run()
phi, psi = rama.results.angles[:, 0, 0], rama.results.angles[:, 0, 1]  # ALA2 only

plt.figure(figsize=(6, 6))
plt.scatter(phi[::plot_stride], psi[::plot_stride], s=4, color="lightgray", alpha=0.3)
for lbl in sorted(set(labels)):
    m = labels == lbl
    color = basin_colors[lbl] if lbl != -1 else "gray"
    plt.scatter(phi[low_en_frame[m]], psi[low_en_frame[m]], marker="o", color=color, zorder=5, alpha=0.6,
                label=f"basin_{lbl}" if lbl != -1 else "unclustered")

# Rough reference locations for the conformational families
plt.scatter([-75], [145], marker="x", color="black", s=100, zorder=6)
plt.annotate("pPII", (-75, 145), textcoords="offset points", xytext=(6, 6),zorder=8)
plt.scatter([-120], [130], marker="x", color="black", s=100, zorder=6)
plt.annotate(r"$\beta$-strand", (-130, 130), textcoords="offset points", xytext=(6, 6),zorder=8)
plt.scatter([-60], [-45], marker="x", color="black", s=100, zorder=6)
plt.annotate(r"RH $\alpha$-helix", (-60, -45), textcoords="offset points", xytext=(6, 6),zorder=8)

plt.axhline(0, color="gray", linewidth=0.5)
plt.axvline(0, color="gray", linewidth=0.5)
plt.xlim(-180, 180)
plt.ylim(-180, 180)
plt.xlabel(r"$\phi$ (°)")
plt.ylabel(r"$\psi$ (°)")
plt.title("Ramachandran plot (ALA2)")
plt.legend()
plt.tight_layout()
plt.savefig('./rama_plot.png', dpi = 800)
plt.show()

## 5.2 Root mean square deviation (RMSD)

RMSD measures how much the peptide's structure deviates, atom-by-atom after an optimal least-squares superposition, from a reference structure — here the first production frame (t = 0). Unlike Rg (a size/compactness measure of a single structure), RMSD directly quantifies conformational change relative to a fixed starting point: a low, stable RMSD means the structure stays close to where it began, while shifts to a different RMSD level often mark a transition to a distinct conformational state.

We compute it for the whole peptide using `MDAnalysis.analysis.rms.RMSD` (which performs the superposition itself). The low-energy representative structures are marked with × (colored by their Section 4 cluster), same convention as the rest of this section.

In [ ]:
from MDAnalysis.analysis import rms

rmsd_protein = rms.RMSD(u, u, select="protein").run().results.rmsd[:, 2]

plt.figure(figsize=(8, 5))
plt.plot(time_ns[::plot_stride], rmsd_protein[::plot_stride], linewidth=0.8, color="lightskyblue", alpha=0.6)
for lbl in sorted(set(labels)):
    m = labels == lbl
    color = basin_colors[lbl] if lbl != -1 else "gray"
    plt.scatter(low_en_time_ns[m], rmsd_protein[low_en_frame[m]], marker="x", color=color, zorder=5,
                label=f"basin_{lbl}" if lbl != -1 else "unclustered")
plt.xlabel("Time (ns)")
plt.ylabel("RMSD (Å)")
plt.legend()
plt.tight_layout()
plt.savefig('RMSD_peptide.png', dpi = 600)
plt.show()

print(f"Peptide RMSD = {rmsd_protein.mean():.3f} ± {rmsd_protein.std():.3f} Å (all frames, relative to t=0)")

## 5.3 Root mean square fluctuation (RMSF)

While RMSD tracks how the *whole* structure drifts from a reference over time, RMSF asks the complementary question per atom: how much does each individual atom move around its own average position over the trajectory? This gives a per-atom flexibility profile rather than a single time series.

Computing RMSF properly requires fitting the trajectory to its own average structure first (not just a single reference frame like t=0), following [MDAnalysis's standard recipe](https://docs.mdanalysis.org/stable/documentation_pages/analysis/rms.html): compute the average structure, align every frame to it, then measure each atom's fluctuation. This alignment is done **in memory on a separate copy of the trajectory** (`u_rmsf`, not `u`) so it doesn't affect any other cell in this notebook — RMSF is the only quantity here that needs the trajectory itself re-aligned rather than just fit-on-the-fly per frame.

We use heavy atoms only (`protein and not name H*`, same convention as the RDF selection), labeled by residue and atom name since there are only 14 of them.

In [ ]:
from MDAnalysis.analysis import align, rms

# Separate universe + its own heavy-atom selection, so aligning it in memory
# below doesn't touch `u` (used by every other cell in this section).
u_rmsf = mda.Universe("prod.tpr", "prod_c.xtc")
heavy_rmsf = u_rmsf.select_atoms("protein and not name H*")

average = align.AverageStructure(u_rmsf, u_rmsf, select="protein and not name H*", ref_frame=0).run()
align.AlignTraj(u_rmsf, average.results.universe, select="protein and not name H*", in_memory=True).run()
rmsf = rms.RMSF(heavy_rmsf).run().results.rmsf

atom_labels = [f"{r}{i}-{n}" for r, i, n in zip(heavy_rmsf.resnames, heavy_rmsf.resids, heavy_rmsf.names)]

plt.figure(figsize=(8, 5))
plt.bar(atom_labels, rmsf, color="tab:blue")
plt.xticks(rotation=90)
plt.ylabel("RMSF (Å)")
plt.tight_layout()
plt.savefig('RMSF_peptide.png', dpi=600)
plt.show()

## 5.4 Radius of gyration

The radius of gyration, $R_g$, measures compactness: larger $R_g$ → more extended conformation, smaller $R_g$ → more compact. We compute it for both the whole peptide and the backbone only, for every frame (MDAnalysis reports distances in Å).

The low-energy representative structures are marked with × (colored by their Section 4 cluster) on the plot below, so their $R_g$ can be read off directly and compared against the general trend (mean ± std. dev., printed underneath) — see the full comparison table in Section 8.4.

In [ ]:
rg_protein = np.empty(len(u.trajectory))
rg_backbone = np.empty(len(u.trajectory))

for i, ts in enumerate(u.trajectory):
    rg_protein[i] = protein.radius_of_gyration()
    rg_backbone[i] = backbone.radius_of_gyration()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8), sharex=True)

ax1.plot(time_ns[::plot_stride], rg_protein[::plot_stride], linewidth=0.8, color="pink", alpha=0.6) #, label="Peptide")
ax2.plot(time_ns[::plot_stride], rg_backbone[::plot_stride], linewidth=0.8, color="gold", alpha=0.6) #, label="Backbone")

for lbl in sorted(set(labels)):
    m = labels == lbl
    color = basin_colors[lbl] if lbl != -1 else "gray"
    cluster_label = f"basin_{lbl}" if lbl != -1 else "unclustered"
    ax1.scatter(low_en_time_ns[m], rg_protein[low_en_frame[m]], marker="x", color=color, zorder=5, label=cluster_label)
    ax2.scatter(low_en_time_ns[m], rg_backbone[low_en_frame[m]], marker="x", color=color, zorder=5, label=cluster_label)

ax1.set_title("Peptide")
ax2.set_title("Backbone")
ax1.set_ylabel(r"$R_g$ (Å)")
ax2.set_ylabel(r"$R_g$ (Å)")
ax2.set_xlabel("Time (ns)")
ax2.sharey(ax1)

handles, legend_labels = ax1.get_legend_handles_labels()
fig.legend(handles, legend_labels, loc="center left", bbox_to_anchor=(1.0, 0.5))
plt.tight_layout()
plt.savefig('./gyr_rad.png',dpi=600, bbox_inches="tight")
plt.show()

print(f"Peptide Rg  = {rg_protein.mean():.3f} ± {rg_protein.std():.3f} Å (all frames)")
print(f"Backbone Rg = {rg_backbone.mean():.3f} ± {rg_backbone.std():.3f} Å (all frames)")

## 5.5 Peptide-water radial distribution function

The radial distribution function g(r) describes hydration structure around the peptide: peaks mark distances where water is preferentially found, g(r) → 1 is bulk-like. We compute it between peptide heavy atoms and water oxygens (`resname SOL and name OW` — verified against this topology's atom names).

g(r) is an ensemble property, not something meaningful for a single frame, so we compute it two ways: once over the whole trajectory (below), and once per basin using each basin's set of low-energy representative frames (Section 4) — a small but physically meaningful sample specific to that conformational family, unlike a lone structure or the whole unclustered trajectory.

In [ ]:
from MDAnalysis.analysis.rdf import InterRDF

rdf_all = InterRDF(protein_heavy, water_O, nbins=100, range=(0.0, 15.0))
rdf_all.run()

plt.figure(figsize=(8, 5))
plt.plot(rdf_all.results.bins, rdf_all.results.rdf, color="black")
plt.axhline(1.0, color="gray", linestyle="--", linewidth=1)
plt.xlim(-0.5,11)
plt.xlabel("r (Å)")
plt.ylabel("g(r)")
plt.tight_layout()
plt.savefig('rdf_PEP_SOL.png', dpi=600)
plt.show()

In [ ]:
# Basin-resolved RDF — each basin's low-energy representative frames (Section 4), one
# InterRDF instance per basin so each is self-normalized by its own frame count.
plt.figure(figsize=(8, 5))
for lbl in sorted(set(labels)):
    frames_b = low_en_frame[labels == lbl]
    rdf_b = InterRDF(protein_heavy, water_O, nbins=100, range=(0.0, 15.0))
    rdf_b.run(frames=frames_b)
    color = basin_colors[lbl] if lbl != -1 else "gray"
    plt.plot(rdf_b.results.bins, rdf_b.results.rdf, color=color, label=f"basin_{lbl} (n={len(frames_b)})")
plt.axhline(1.0, color="gray", linestyle="--", linewidth=1)
plt.xlabel("r (Å)")
plt.ylabel("g(r)")
plt.xlim(-0.5,11)
plt.legend()
plt.tight_layout()
plt.show()

## 5.6 Peptide-water hydrogen bonds

We count hydrogen bonds formed **between the peptide and water only** (excluding peptide-peptide and water-water) using MDAnalysis's `HydrogenBondAnalysis`, with the standard geometric criteria: donor-acceptor distance ≤ 3.0 Å, donor-hydrogen-acceptor angle ≥ 150°.

We compute this two ways: **All** (every protein donor/acceptor — termini included) and **Backbone** (restricted to the mainchain amide N-H and carbonyl C=O). For this topology, `protein and name H` isolates exactly the two standard backbone amide hydrogens, correctly excluding the N-terminal `H1/H2/H3` (charged NH3⁺) and the C-terminal `HO` (neutral COOH); `backbone and name O*` gives the three backbone carbonyl oxygens, excluding the C-terminal `OT` (both verified directly against this topology's atom names).

As with Rg, the low-energy representative structures are marked on the time series below (colored by cluster, as in 5.1) and compared against the general trend in Section 5.5.

In [ ]:
from MDAnalysis.analysis.hydrogenbonds.hbond_analysis import HydrogenBondAnalysis


def hbond_counts_per_frame(hba, frame_indices):
    """Per-frame hbond counts for an arbitrary (possibly non-contiguous) frame list.
    HydrogenBondAnalysis.count_by_time() assumes a contiguous start/step range and
    breaks on scattered frame subsets, so we do it manually."""
    counts = np.zeros(len(frame_indices), dtype=int)
    pos = {f: i for i, f in enumerate(frame_indices)}
    fr, c = np.unique(hba.results.hbonds[:, 0].astype(int), return_counts=True)
    for f, n in zip(fr, c):
        counts[pos[f]] = n
    return counts


hba = HydrogenBondAnalysis(
    universe=u,
    hydrogens_sel="(protein and name H*) or (resname SOL and name HW*)",
    acceptors_sel="(protein and name O*) or (resname SOL and name OW)",
    between=["protein", "resname SOL"],
    d_a_cutoff=3.0,
    d_h_a_angle_cutoff=150,
    update_selections=False,
)
hba.run()  # full trajectory, ~1-2 min

hba_bb = HydrogenBondAnalysis(
    universe=u,
    hydrogens_sel="(protein and name H) or (resname SOL and name HW*)",
    acceptors_sel="(backbone and name O*) or (resname SOL and name OW)",
    between=["protein", "resname SOL"],
    d_a_cutoff=3.0,
    d_h_a_angle_cutoff=150,
    update_selections=False,
)
hba_bb.run()  # backbone-only donor/acceptor on the protein side, ~1-2 min

n_hbonds = hbond_counts_per_frame(hba, np.arange(len(u.trajectory)))
n_hbonds_bb = hbond_counts_per_frame(hba_bb, np.arange(len(u.trajectory)))

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8), sharex=True)
ax1.plot(time_ns[::plot_stride], n_hbonds[::plot_stride], color="cyan", alpha=0.3, linewidth=0.8)
ax2.plot(time_ns[::plot_stride], n_hbonds_bb[::plot_stride], color="orange", alpha=0.3, linewidth=0.8)

for lbl in sorted(set(labels)):
    m = labels == lbl
    color = basin_colors[lbl] if lbl != -1 else "gray"
    cluster_label = f"basin_{lbl}" if lbl != -1 else "unclustered"
    ax1.scatter(low_en_time_ns[m], n_hbonds[low_en_frame[m]], marker="x", color=color, zorder=5, label=cluster_label)
    ax2.scatter(low_en_time_ns[m], n_hbonds_bb[low_en_frame[m]], marker="x", color=color, zorder=5, label=cluster_label)

ax1.set_title("Peptide")
ax2.set_title("Backbone")
ax1.set_ylabel("Peptide-water H-bonds")
ax2.set_ylabel("Peptide-water H-bonds")
ax2.set_xlabel("Time (ns)")

handles, legend_labels = ax1.get_legend_handles_labels()
fig.legend(handles, legend_labels, loc="center left", bbox_to_anchor=(1.0, 0.5))
plt.tight_layout()
plt.savefig('H_bonds.png',dpi=600, bbox_inches="tight")
plt.show()

print(f"Peptide      H-bonds = {n_hbonds.mean():.2f} ± {n_hbonds.std():.2f} (all frames)")
print(f"Backbone H-bonds = {n_hbonds_bb.mean():.2f} ± {n_hbonds_bb.std():.2f} (all frames)")

## 5.7 Solvent-accessible surface area (SASA)

SASA complements the H-bond count: it measures how much of the peptide's surface is exposed to solvent overall (a geometric measure), rather than counting specific directional interactions. A more compact conformation generally buries more surface and has lower SASA.

MDAnalysis has no built-in SASA calculator, so this uses the `freesasa` package (`pip install freesasa`) — for each frame, the current `protein` coordinates are written to a temporary PDB and passed to `freesasa.calc`. `freesasa.setVerbosity(freesasa.silent)` suppresses its per-atom warnings about the non-standard `OT` atom name (the C-terminal carboxyl oxygen from `pdb2gmx`'s neutral-COOH naming, from the tutorial for GAG⁺'s topology) — freesasa still correctly guesses it as an oxygen with a standard radius.

Computed over every frame, as with Rg/H-bonds (~1 min), with the low-energy representative structures marked the same way.

In [ ]:
import freesasa
import tempfile, os

freesasa.setVerbosity(freesasa.silent)


def frame_sasa(frame_index):
    u.trajectory[frame_index]
    with tempfile.NamedTemporaryFile(suffix=".pdb", delete=False) as tmp:
        protein.write(tmp.name)
        tmp_path = tmp.name
    try:
        return freesasa.calc(freesasa.Structure(tmp_path)).totalArea()
    finally:
        os.remove(tmp_path)


sasa = np.array([frame_sasa(i) for i in range(len(u.trajectory))])  # full trajectory, ~1 min

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(time_ns[::plot_stride], sasa[::plot_stride], linewidth=0.8, color="turquoise", alpha=0.4)
for lbl in sorted(set(labels)):
    m = labels == lbl
    color = basin_colors[lbl] if lbl != -1 else "gray"
    plt.scatter(low_en_time_ns[m], sasa[low_en_frame[m]], marker="x", color=color, zorder=5,
                label=f"basin_{lbl}" if lbl != -1 else "unclustered")
plt.xlabel("Time (ns)")
plt.ylabel(r"SASA (Å$^2$)")
plt.legend()
plt.tight_layout()
plt.savefig('SASA.png',dpi=600)
plt.show()

print(f"SASA = {sasa.mean():.1f} ± {sasa.std():.1f} Å² (all frames)")

## 5.8 Connect the structural descriptors with the free-energy landscape

For each low-energy representative structure (Section 4), this table combines its position in the landscape (proj1, proj2, ΔG) with its Rg, H-bond count, and SASA, grouped by basin and ranked from lowest to highest ΔG within each — directly comparable to the "all frames" mean ± std. dev. reported in 5.1/5.3/5.4. The scatter plot then shows every trajectory frame colored by $R_g$, with the representative structures marked on top.

In [ ]:
print(f"{'basin':>5} {'time (ps)':>10} {'dG (J/mol)':>12} {'proj1':>9} {'proj2':>9} {'Rg protein (Å)':>15} {'Rg backbone (Å)':>16} {'H-bonds':>8} {'H-bonds bb':>10} {'SASA (Å²)':>10}")
for lbl in sorted(set(labels)):
    idx = np.where(labels == lbl)[0]
    idx = idx[np.argsort(low_en_dG[idx])]  # increasing dG within this basin
    for k in idx:
        fr = low_en_frame[k]
        #de-comment if you want to print the values for each low energy conformation we extracted
        #print(f"{lbl:5d} {low_en_time[k]:10.0f} {low_en_dG[k]:12.2f} {low_en_p1[k]:9.4f} {low_en_p2[k]:9.4f} "
        #      f"{rg_protein[fr]:15.3f} {rg_backbone[fr]:16.3f} {n_hbonds[fr]:8d} {n_hbonds_bb[fr]:10d} {sasa[fr]:10.1f}")

print()

print("per-basin stats (over this basin's low-energy representative structures):")
for lbl in sorted(set(labels)):
    idx = np.where(labels == lbl)[0]
    fr = low_en_frame[idx]
    rp, rb, hb, hbbb, sa = rg_protein[fr], rg_backbone[fr], n_hbonds[fr], n_hbonds_bb[fr], sasa[fr]
    print(f"basin {lbl} (n={len(idx)}):")
    print(f"  Rg protein  = {rp.mean():.3f} ± {rp.std():.3f} Å  [min {rp.min():.3f}, max {rp.max():.3f}]")
    print(f"  Rg backbone = {rb.mean():.3f} ± {rb.std():.3f} Å  [min {rb.min():.3f}, max {rb.max():.3f}]")
    print(f"  H-bonds     = {hb.mean():.2f} ± {hb.std():.2f}    [min {hb.min()}, max {hb.max()}]")
    print(f"  H-bonds bb  = {hbbb.mean():.2f} ± {hbbb.std():.2f}    [min {hbbb.min()}, max {hbbb.max()}]")
    print(f"  SASA        = {sa.mean():.1f} ± {sa.std():.1f} Å²  [min {sa.min():.1f}, max {sa.max():.1f}]")

print()
print(f"all-frame trend: Rg protein = {rg_protein.mean():.3f} ± {rg_protein.std():.3f} Å, "
      f"Rg backbone = {rg_backbone.mean():.3f} ± {rg_backbone.std():.3f} Å, "
      f"H-bonds = {n_hbonds.mean():.2f} ± {n_hbonds.std():.2f}, "
      f"H-bonds bb = {n_hbonds_bb.mean():.2f} ± {n_hbonds_bb.std():.2f}, "
      f"SASA = {sasa.mean():.1f} ± {sasa.std():.1f} Å²")